# Tutorial 3: symmetric slabs, surface formation energies and their dependence on the chemical potential

Symmetric Na3OCl(100) slabs have two terminations (Na2O- and NaCl-terminated). They are not stoichiometric, so their
stability is given by the surface formation energy (SFE), which depends on the chemical potentials. The decomposition
products Na and NaCl (`decomps1`) set the Na-rich limit. With `--sfe_scan1 Na,Cl,-1.5,0` the SFE is also computed as a
function of $\Delta\mu_{Na}$ (with $\mu_{Cl}$ compensating, so that the bulk stays in equilibrium).
MACE-MP-0 small, about 2 minutes on a CPU.

In [ ]:
import os
import warnings

for w in ("Environment variable TORCH_FORCE", "`torch.jit", "Occupancies"):   # harmless notes of torch/ASE
    warnings.filterwarnings("ignore", message=w)
RUN = not os.path.exists("OUTPUT/sfe_vs_mu_1.csv")   # set to True to re-run
if RUN:
    from interfacer.cli import main
    main(["--input", "input.toml", "--sfe_scan1", "Na,Cl,-1.5,0.0", "-q"])

## Results

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
from ase.io import read
from ase.visualize.plot import plot_atoms

J = 0.160217663   # eV/nm^2 -> J/m^2
res = json.load(open("OUTPUT/results.json"))
print("Bulk energies: %.4f and %.4f eV/atom" % (res["E_bulk1"], res["E_bulk2"]))
print("Surfaces:", res["miller1"], "|", res["miller2"])
for n in (1, 2):
    for s in res["slabs%d" % n]:
        print("  slab %d %-12s %3d atoms  SFE = %6.2f eV/nm^2 = %.2f J/m^2" % (n, s["name"], s["natoms"], s["sfe"], s["sfe"] * J))
for r in res["interfaces"]:
    print("\nInterface %s (%s | %s)" % (r["name"], r["slab1"], r["slab2"]))
    print("  misfit %.2f%% -> %.2f%% after alignment, area %.1f A^2" % (100 * r["misfit_before"], 100 * r["misfit_after"], r["area"]))
    print("  W_surf = %.2f and %.2f J/m^2, separation %.2f A, %d interface(s) in the cell"
          % (r["W_surf1"] * J, r["W_surf2"] * J, r["sep"], r["n_interfaces"]))
    print("  W_ad = %.2f J/m^2, gamma_int = %.2f J/m^2" % (r["W_ad"] * J, r["gamma_int"] * J))

## SFE as a function of the Na chemical potential

The termination with the lowest SFE is the stable one; the crossing gives the chemical potential at which the stable
termination changes.

In [ ]:
d = np.loadtxt("OUTPUT/sfe_vs_mu_1.csv", delimiter=",", skiprows=1)
names = open("OUTPUT/sfe_vs_mu_1.csv").readline().strip().split(",")[1:]
labels = {s["name"]: "%s (%s)" % (s["name"], read("OUTPUT/%s_preopted.xyz" % s["name"]).get_chemical_formula(empirical=False))
          if os.path.exists("OUTPUT/%s_preopted.xyz" % s["name"]) else s["name"] for s in res["slabs1"]}
plt.figure(figsize=(5, 3.5))
for k, n in enumerate(names):
    plt.plot(d[:, 0], d[:, k + 1] * J, label=labels.get(n, n))
plt.xlabel(r"$\Delta\mu_{Na}$ (eV)  (Na-poor $\leftarrow$ $\rightarrow$ Na-rich)")
plt.ylabel("surface formation energy (J/m$^2$)")
plt.legend()
plt.tight_layout()
stable = np.argmin(d[:, 1:], axis=1)
for k in sorted(set(stable)):
    mu = d[stable == k, 0]
    print("%s is the most stable termination for %.2f <= dmu_Na <= %.2f eV" % (names[k], mu.min(), mu.max()))

## Interfaces of both terminations with Na(100)

`no_slabs = 2` builds an interface with each termination.

In [ ]:
for r in res["interfaces"]:
    print("%-28s %-12s W_ad = %5.2f J/m^2   gamma_int = %5.2f J/m^2" % (r["name"], r["slab1"], r["W_ad"] * J, r["gamma_int"] * J))
fig, axes = plt.subplots(1, len(res["interfaces"]), figsize=(4.5 * len(res["interfaces"]), 5))
for ax, r in zip(np.atleast_1d(axes), res["interfaces"]):
    plot_atoms(read("OUTPUT/%s_opted.xyz" % r["name"]), ax, rotation="-90x", radii=0.8)
    ax.set_title(r["slab1"]); ax.set_axis_off()
plt.tight_layout()